## Project: Credit Risk Modeling ML Pipeline

This project is a binary classification task aimed at predicting whether a person has experienced 90 days or more of past-due delinquency.The dataset includes financial and demographic features.Using a variety of classification algorithms and evaluation metrics, it aims to identify the most effective approach for predicting credit default events.The notebook includes data preprocessing, missing value handling, model training, performance evaluation, and visualization.

The related dataset can be found on Kaggle:
https://www.kaggle.com/c/GiveMeSomeCredit/data

#### Key Features:

- SeriousDlqin2yrs: Person experienced 90 days past due delinquency or worse
- RevolvingUtilizationOfUnsecuredLines: Total balance on credit cards and personal lines of credit except real estate and no installment debt like car loans divided by the sum of credit limits
- age: Age of borrower in years
- NumberOfTime30-59DaysPastDueNotWorse: Number of times borrower has been 30-59 days past due but no worse in the last 2 years
- DebtRatio: Monthly debt payments, alimony, living costs divided by monthly gross income
- MonthlyIncome: Monthly income
- NumberOfOpenCreditLinesAndLoans: Number of open loans and lines of credit
- NumberOfTimes90DaysLate: Number of times borrower has been 90 days or more past due
- NumberRealEstateLoansOrLines: Number of mortgage and real estate loans including home equity lines of credit
- NumberOfTime60-89DaysPastDueNotWorse: Number of times borrower has been 60-89 days past due but no worse in the last 2 years
- NumberOfDependents: Number of dependents in family excluding themselves

### ML Pipeline Overview

1. Data Ingestion (raw credit data)
2. Data Cleaning & Preprocessing
3. Feature Engineering (handling missing values, outliers)
4. Model Training (Logistic Regression, Random Forest, XGBoost)
5. Model Evaluation (ROC-AUC, Precision, Recall， F1-score)
6. Model Selection (best-performing model chosen for deployment)
7. Model Persistence (saving trained model and scaler as .pkl artifacts)
8. API Inference Service (FastAPI-based real-time credit risk scoring endpoint)
9. Prediction Logging & Monitoring (logging inputs, outputs, and model version for traceability)

In [ ]:
import os
import csv
import joblib
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from fastapi import FastAPI
from datetime import datetime
from pydantic import BaseModel
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.metrics import precision_score, recall_score, f1_score
#pip install xgboost

### Data Loading

In [ ]:
def load_data(path):
    df = pd.read_csv(path)
    df = df.copy()
    return df

### EDA

In [ ]:
def perform_eda(df):
    print('Data shape: ', df.shape)
    print('------------------------------------------------------------------------------')
    print(df.info())
    print('------------------------------------------------------------------------------')
    print(df.describe())

### Preprocessing

In [ ]:
def preprocess_data(df):
    df = df.drop('Unnamed: 0', axis=1)

    df['Income_dummy'] = df['MonthlyIncome'].isnull().astype(int)
    df['MonthlyIncome'] = df['MonthlyIncome'].fillna(df['MonthlyIncome'].median())
    df['NumberOfDependents'] = df['NumberOfDependents'].fillna(df['NumberOfDependents'].median())

    # cap outliers
    income_99 = df['MonthlyIncome'].quantile(0.99)
    debt_99 = df['DebtRatio'].quantile(0.99)

    df['MonthlyIncome'] = df['MonthlyIncome'].clip(upper=income_99)
    df['DebtRatio'] = df['DebtRatio'].clip(upper=debt_99)

    return df

### Data Split + Scaling

In [ ]:
def split_data(df):
    X = df.drop('SeriousDlqin2yrs', axis=1)
    y = df['SeriousDlqin2yrs']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, stratify=y, random_state=42
    )

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    return X_train, X_test, y_train, y_test, scaler

### Train Models

In [ ]:
def train_models(X_train, y_train):
    log_clf = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
    rf_clf = RandomForestClassifier(n_estimators=400, max_depth=10, class_weight='balanced', random_state=42)

    scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
    xgb_clf = XGBClassifier(scale_pos_weight=scale_pos_weight, random_state=42)

    voting_clf = VotingClassifier(
        estimators=[('log', log_clf), ('rf', rf_clf), ('xgb', xgb_clf)],
        voting='soft'
    )

    models = {
        "log": log_clf,
        "rf": rf_clf,
        "xgb": xgb_clf,
        "voting": voting_clf
    }

    for model in models.values():
        model.fit(X_train, y_train)

    return models

### Evaluation via Metrics

In [ ]:
def evaluate_models(models, X_test, y_test):
    results = {}

    for name, model in models.items():
        pred = model.predict(X_test)
        prob = model.predict_proba(X_test)[:, 1]

        results[name] = {
            "roc_auc": roc_auc_score(y_test, prob),
            "precision": precision_score(y_test, pred),
            "recall": recall_score(y_test, pred)
             "f1_score": f1_score(y_test, pred)
        }

    return results

In [ ]:
def main():
    df = load_data('GiveMeSomeCredit-training.csv')
    df = preprocess_data(df)

    X_train, X_test, y_train, y_test, scaler = split_data(df)
    joblib.dump(scaler, "scaler.pkl")

    models = train_models(X_train, y_train)
    results = evaluate_models(models, X_test, y_test)

    best_model = models["xgb"]
    joblib.dump(best_model, "model.pkl")

    print(results)


if __name__ == "__main__":
    main()

### Predict using API inference endpoint

In a production environment, the API inference service would typically be maintained as a separate module (e.g., api.py) to ensure clear separation between model training and model serving.
For the purpose of this project demonstration and to provide a complete end-to-end view of the system, the inference logic is included here for reference.

#### Deploy Endpoint

In [ ]:
# ===========================
# Load solar model and scaler
# ===========================
model = joblib.load("model.pkl")
scaler = joblib.load("scaler.pkl")

MODEL_VERSION = "CREDIT_RISK_v1_2026_04"
LOG_FILE = "prediction_log.csv"

# ===========================
# Initialize log
# ===========================
if not os.path.exists(LOG_FILE):
    with open(LOG_FILE, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "timestamp",
            "features",
            "prediction",
            "probability",
            "model_version"
        ])

In [ ]:
# ===========================
# Request format
# ===========================

class InputData(BaseModel):
    features: list[float]

#### Initialize FastAPI

In [ ]:
app = FastAPI(title="Credit Risk API")

#### Health Check

In [ ]:
@app.get("/health")
def health():
    return {"status": "ok"}

#### Predict Endpoint

In [ ]:
@app.post("/predict")
def predict(data: InputData):

    x = np.array(data.features).reshape(1, -1)
    x_scaled = scaler.transform(x)

    prob = model.predict_proba(x_scaled)[0][1]
    pred = int(prob > 0.5)

    # log: for future drift monitoring
    with open(LOG_FILE, "a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            datetime.now().isoformat(),
            data.features,
            float(pred),
            float(prob),
            MODEL_VERSION
        ])

    return {
        "prediction": pred,
        "probability": float(prob),
        "model_version": MODEL_VERSION
    }

#### Run Real-time Credit Risk Scoring API

In [ ]:
if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8001, reload=True)